[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedorShind/EMRG/blob/main/docs/tutorials/qaoa_maxcut_mitigation.ipynb)

# Mitigating QAOA Errors for MaxCut with EMRG

QAOA circuits for MaxCut are deeper and noisier than typical VQE ansatze because each graph edge contributes an RZZ gate to the cost layer. EMRG adapts its technique selection to these circuit characteristics, choosing between ZNE, PEC, CDR, and composite ZNE-over-PEC recipes. This notebook builds a 5-node MaxCut QAOA circuit, analyzes it, and compares the available techniques.

In [ ]:
!pip install -q emrg[preview]

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

from emrg import analyze_circuit, generate_recipe

## Build the QAOA circuit

QAOA alternates cost layers (RZZ for each graph edge, encoding the MaxCut objective) and mixer layers (RX on each qubit, providing exploration). The graph has 5 nodes and 6 edges; we use 2 QAOA layers (p=2).

In [ ]:
rng = np.random.default_rng(42)

n_nodes = 5
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
n_layers = 2

qc = QuantumCircuit(n_nodes, n_nodes)

# Initial superposition
for q in range(n_nodes):
    qc.h(q)

for layer in range(n_layers):
    # Cost layer: RZZ for each edge
    gamma = float(rng.uniform(0, 2 * np.pi))
    for i, j in edges:
        qc.rzz(gamma, i, j)

    # Mixer layer: RX on each qubit
    beta = float(rng.uniform(0, 2 * np.pi))
    for q in range(n_nodes):
        qc.rx(beta, q)

qc.measure(range(n_nodes), range(n_nodes))

print(f"Nodes: {n_nodes}, Edges: {len(edges)}")
print(f"QAOA layers: {n_layers}")
print(f"Qubits: {qc.num_qubits}")
print(f"Depth:  {qc.depth()}")

## Analyze the circuit

This circuit has 12 RZZ gates and 10 RX gates, all at non-Clifford angles. Compare this to the H2 VQE ansatz (8 RY gates, 6 CX, depth 8).

In [ ]:
features = analyze_circuit(qc)

print(f"Depth:               {features.depth}")
print(f"Total gates:         {features.total_gate_count}")
print(f"Multi-qubit gates:   {features.multi_qubit_gate_count}")
print(f"Noise estimate:      {features.estimated_noise_factor}")
print(f"Noise category:      {features.noise_category}")
print(f"Non-Clifford count:  {features.non_clifford_count}")
print(f"Non-Clifford frac:   {features.non_clifford_fraction:.4f}")
print(f"PEC overhead est:    {features.pec_overhead_estimate:.2f}")
print(f"Layer heterogeneity: {features.layer_heterogeneity:.4f}")

Key differences from the H2 circuit: depth 13 (vs 8), 81% non-Clifford fraction (vs 57%), and PEC overhead of about 25x (vs about 2.3x). The depth is above CDR's minimum threshold of 10 and below its maximum of 40, and the non-Clifford fraction exceeds the 20% threshold. EMRG selects CDR here. Composite is available, but CDR wins because this circuit's dominant risk is non-Clifford structure rather than a moderate-depth Clifford-heavy noise profile.

## Generate the default recipe

In [ ]:
result = generate_recipe(qc, explain=True)

kwargs = result.recipe.factory_kwargs
print(f"Technique:          {result.recipe.technique}")
if result.recipe.technique == "cdr":
    print(f"Training circuits:  {kwargs.get('num_training_circuits')}")
    print(f"Fit method:         {kwargs.get('fit_method')}")
else:
    print(f"Factory:            {result.recipe.factory_name}")
    print(f"Scale factors:      {result.recipe.scale_factors}")
print()
print(result.code)

EMRG selected CDR because 81% of this circuit's gates are non-Clifford and the depth (13) falls within CDR's range (10 to 40). CDR replaces non-Clifford gates with Clifford substitutes to create training circuits that a classical simulator can evaluate exactly. A linear regression model fitted on the training data then corrects the noisy result.

## Compare all four techniques

Force each technique and compare the generated configuration. This is not saying each one is equally appropriate for QAOA; it shows what EMRG will generate when you ask for a specific mitigation family.

In [ ]:
for tech in ["zne", "pec", "cdr", "composite"]:
    kwargs = {"technique": tech}
    if tech in {"pec", "composite"}:
        kwargs["noise_model_available"] = True
    r = generate_recipe(qc, **kwargs)
    fk = r.recipe.factory_kwargs

    print(f"=== {tech.upper()} ===")
    if tech == "zne":
        print(f"  Factory:       {r.recipe.factory_name}")
        print(f"  Scale factors: {r.recipe.scale_factors}")
        print(f"  Scaling:       {r.recipe.scaling_method}")
        print(f"  Overhead:      {r.recipe.estimated_overhead:.0f}x")
    elif tech == "pec":
        print(f"  Samples:       {fk.get('num_samples')}")
        print(f"  Overhead:      {r.recipe.estimated_overhead:.1f}x")
    elif tech == "cdr":
        n_train = fk.get("num_training_circuits")
        print(f"  Training:      {n_train} circuits")
        print(f"  Fit method:    {fk.get('fit_method')}")
        print(f"  Overhead:      {r.recipe.estimated_overhead:.0f}x")
    elif tech == "composite":
        zne_component, pec_component = r.recipe.components
        factory = zne_component.factory_name
        scales = zne_component.scale_factors
        zne_detail = f"{factory}, scales {scales}"
        print(f"  ZNE:           {zne_detail}")
        print(f"  PEC samples:   {pec_component.factory_kwargs.get('num_samples')}")
        print(f"  Overhead:      {r.recipe.estimated_overhead:.1f}x")
    print()

ZNE uses LinearFactory with 3 scale factors (3x overhead). PEC uses samples derived from the estimated PEC cost. CDR uses 12 training circuits for this QAOA example. Composite multiplies the outer ZNE scale-factor cost by the inner PEC sampling cost, so it is powerful but expensive. CDR remains the default here because it targets the non-Clifford gates directly.

## When composite is selected automatically

The QAOA circuit above is CDR-heavy. To see the new automatic composite path, use a small Clifford-heavy circuit with moderate depth and a noise model. This is a decision-engine example, not a MaxCut benchmark.

In [ ]:
composite_candidate = QuantumCircuit(4, 4)
for _ in range(16):
    composite_candidate.cx(0, 1)
composite_candidate.measure(range(4), range(4))

composite_result = generate_recipe(
    composite_candidate,
    noise_model_available=True,
)

print(f"Depth:      {composite_result.features.depth}")
print(f"Noise:      {composite_result.features.noise_category}")
print(f"PEC cost:   {composite_result.features.pec_overhead_estimate:.1f}x")
print(f"Technique:  {composite_result.recipe.technique}")
print(f"Overhead:   {composite_result.recipe.estimated_overhead:.1f}x")
print("Components:")
for component in composite_result.recipe.components:
    print(f"  - {component.technique.upper()}")

This is the composite sweet spot: enough depth for extrapolation to matter, a noise model for PEC, no strong CDR signal, and a combined overhead under the 1000x auto-selection cutoff.

## Preview mode

Preview is optional for this QAOA circuit. The generated CDR recipe is still useful, but the local preview path depends on Mitiq's Qiskit-to-Cirq conversion. Some RZZ-heavy circuits may be skipped by the simulator; the cell below prints a clear warning when that happens. Requires `pip install emrg[preview]`.

In [ ]:
preview_result = generate_recipe(qc, preview=True, noise_level=0.01)

if preview_result.preview and preview_result.preview.ideal_value is not None:
    p = preview_result.preview
    print(f"Technique:  {p.technique}")
    print(f"Ideal:      {p.ideal_value:+.4f}")
    print(f"Noisy:      {p.noisy_value:+.4f}  (error: {p.noisy_error:.4f})")
    print(f"Mitigated:  {p.mitigated_value:+.4f}  (error: {p.mitigated_error:.4f})")
    print(f"Reduction:  {p.error_reduction:.1f}x")
else:
    p = preview_result.preview
    warning = p.warning if p else "cirq not installed"
    print(f"Preview skipped: {warning}")

## Summary

Different circuits get different recommendations. The H2 VQE ansatz (depth 8, 57% non-Clifford) gets ZNE by default because it is too shallow for CDR or automatic composite. This QAOA circuit (depth 13, 81% non-Clifford) gets CDR because the depth and non-Clifford fraction both exceed CDR's activation thresholds. A moderate-depth, Clifford-heavy circuit with a noise model can get composite ZNE-over-PEC automatically when the combined overhead stays below the cutoff.

- [EMRG on GitHub](https://github.com/FedorShind/EMRG)
- [EMRG on PyPI](https://pypi.org/project/emrg/)
- [Mitiq documentation](https://mitiq.readthedocs.io/)